# Lab 2 — rozwiązania

**Zadanie 1.** Dask przy `read_csv` nie ogląda całego pliku. Typy zgaduje z próbki (domyślnie pierwsze 10 wierszy). Braki wstawione dalej psują to zgadywanie dopiero przy `.compute()`.

Oryginalny plik. `idZamowienia` jest całkowite.

In [1]:
import os

import dask.dataframe as dd
import pandas as pd

src = os.path.join("..", "lab_01", "zamowienia.csv")
df = pd.read_csv(src, sep=";")
df.dtypes

Kraj                   str
Sprzedawca             str
Data zamowienia        str
idZamowienia         int64
Utarg              float64
dtype: object

Po trzy braki w każdej kolumnie, wszystkie poza pierwszymi 10 wierszami. `Int64` zostawia w CSV puste pole, a nie zamienia całej kolumny na `10248.0`.

In [2]:
df["idZamowienia"] = df["idZamowienia"].astype("Int64")
braki = {
    "Kraj": [15, 30, 45],
    "Sprzedawca": [16, 31, 46],
    "Data zamowienia": [17, 32, 47],
    "idZamowienia": [20, 35, 50],
    "Utarg": [21, 36, 51],
}
for kolumna, wiersze in braki.items():
    df.loc[wiersze, kolumna] = pd.NA

out = "zamowienia_missing.csv"
df.to_csv(out, sep=";", index=False)
df.loc[15:21, ["Kraj", "Sprzedawca", "Data zamowienia", "idZamowienia", "Utarg"]]

,Kraj,Sprzedawca,Data zamowienia,idZamowienia,Utarg
15,NaN,Dudek,2003-07-31,10263,1873.80
16,Polska,NaN,2003-08-23,10264,695.62
17,Niemcy,Fuller,NaN,10265,1176.00
18,Niemcy,Leverling,2003-07-31,10266,346.56
19,Niemcy,Peacock,2003-08-06,10267,3536.60
20,Niemcy,Callahan,2003-08-02,<NA>,1101.20
21,Polska,Kowalski,2003-08-09,10269,NaN


Dask zgaduje typy z początku pliku. Brak w `idZamowienia` jest dopiero w wierszu 20, więc próbka nadal widzi `int64` — tak jak oryginał.

In [3]:
ddf = dd.read_csv(out, sep=";")
ddf.dtypes

Kraj                string
Sprzedawca          string
Data zamowienia     string
idZamowienia         int64
Utarg              float64
dtype: object

`.compute()` czyta cały plik. Puste `idZamowienia` jest `float64`, a próbka zapowiedziała `int64`, więc Dask przerywa.

In [4]:
ddf["idZamowienia"].sum().compute()

ValueError: Mismatched dtypes found in `pd.read_csv`/`pd.read_table`.

+--------------+---------+----------+
| Column       | Found   | Expected |
+--------------+---------+----------+
| idZamowienia | float64 | int64    |
+--------------+---------+----------+

Usually this is due to dask's dtype inference failing, and
*may* be fixed by specifying dtypes manually by adding:

dtype={'idZamowienia': 'float64'}

to the call to `read_csv`/`read_table`.

Alternatively, provide `assume_missing=True` to interpret
all unspecified integer columns as floats.

`sample_rows=10` — brak nadal poza próbką, ten sam błąd.

In [5]:
proba_10 = dd.read_csv(out, sep=";", sample_rows=10)
print(proba_10.dtypes["idZamowienia"])
proba_10["idZamowienia"].sum().compute()

int64


ValueError: Mismatched dtypes found in `pd.read_csv`/`pd.read_table`.

+--------------+---------+----------+
| Column       | Found   | Expected |
+--------------+---------+----------+
| idZamowienia | float64 | int64    |
+--------------+---------+----------+

Usually this is due to dask's dtype inference failing, and
*may* be fixed by specifying dtypes manually by adding:

dtype={'idZamowienia': 'float64'}

to the call to `read_csv`/`read_table`.

Alternatively, provide `assume_missing=True` to interpret
all unspecified integer columns as floats.

`sample_rows=25` — próbka sięga wiersza 20, Dask od razu ustawia `float64` i suma przechodzi.

In [6]:
proba_25 = dd.read_csv(out, sep=";", sample_rows=25)
print(proba_25.dtypes["idZamowienia"])
proba_25["idZamowienia"].sum().compute()

float64


np.float64(8476244.0)

`sample_rows=100` — to samo co przy 25: brak jest w próbce, typ jest od razu poprawny.

In [7]:
proba_100 = dd.read_csv(out, sep=";", sample_rows=100)
print(proba_100.dtypes["idZamowienia"])
proba_100["idZamowienia"].sum().compute()

float64


np.float64(8476244.0)

Porównanie na końcu. **Przed** `.compute()` to typy zgadnięte z 10 wierszy. **Po** to typy prawdziwej ramki, gdy próbka ma 100 wierszy i obliczenie się udaje. Różnica jest w `idZamowienia`: `int64` przed, `float64` po, bo brak nie mieści się w liczbie całkowitej.

In [8]:
przed = dd.read_csv(out, sep=";", sample_rows=10).dtypes
po = dd.read_csv(out, sep=";", sample_rows=100).compute().dtypes

pd.DataFrame({"przed compute": przed.astype(str), "po compute": po.astype(str)})

,przed compute,po compute
Kraj,string,string
Sprzedawca,string,string
Data zamowienia,string,string
idZamowienia,int64,float64
Utarg,float64,float64


**Zadanie 2.** Plugin dashboardu Daska dla Jupyter Lab jest zainstalowany (`dask-labextension` 7.0.0, [dokumentacja](https://docs.dask.org/en/stable/dashboard.html)).

W terminalu, z tego samego Pythona co kernel:

```text
python -m jupyter lab
```

W Jupyter Lab ikona Daska jest na lewym pasku. Wklej tam adres z `client.dashboard_link`, potem uruchom komórkę testową. Na wykresie Task Stream i Progress widać lecące zadania.

In [9]:
import json
import socket
import time
from urllib.request import urlopen

from dask.distributed import Client, get_client
import dask.array as da

def port_wolny(port):
    with socket.socket() as s:
        s.settimeout(0.2)
        return s.connect_ex(("127.0.0.1", port)) != 0

def zamknij_klaster():
    try:
        client.shutdown()
    except Exception:
        pass
    try:
        get_client().shutdown()
    except Exception:
        pass
    if not port_wolny(8787):
        try:
            with urlopen("http://127.0.0.1:8787/json/identity.json", timeout=2) as resp:
                adres = json.load(resp)["address"]
            Client(adres).shutdown()
        except Exception:
            pass
    for _ in range(20):
        if port_wolny(8787):
            return
        time.sleep(0.25)
    raise RuntimeError(
        "Port 8787 jest dalej zajęty przez stary klaster. Zrestartuj kernel i odpal zadania od początku."
    )

zamknij_klaster()
client = Client(n_workers=4, memory_limit="1GiB")
print(client.dashboard_link)
client

http://127.0.0.1:8787/status


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 16,Total memory: 4.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:53814,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:53834,Total threads: 4
Dashboard: http://127.0.0.1:53838/status,Memory: 1.00 GiB
Nanny: tcp://127.0.0.1:53817,


In [ ]:
x = da.ones((20_000_000,), chunks=(1_000_000,))
x.sum().compute()

**Zadanie 3.** Komórka gasi klaster z zadania 2 i stawia jeden nowy: zostaje co najmniej 8 GB RAM i 2 rdzenie. Następna komórka wczytuje na nim pliki parquet, jeden plik na worker, bez zmiany typów. Czas jest mierzony aż `wait` potwierdzi, że dane są w pamięci. Zadania 4 używają tego samego klastra.

In [3]:
import json
import os
import socket
import time
from urllib.request import urlopen

import psutil
from dask.distributed import Client, get_client

def port_wolny(port):
    with socket.socket() as s:
        s.settimeout(0.2)
        return s.connect_ex(("127.0.0.1", port)) != 0

def zamknij_klaster():
    try:
        client.shutdown()
    except Exception:
        pass
    try:
        get_client().shutdown()
    except Exception:
        pass
    if not port_wolny(8787):
        try:
            with urlopen("http://127.0.0.1:8787/json/identity.json", timeout=2) as resp:
                adres = json.load(resp)["address"]
            Client(adres).shutdown()
        except Exception:
            pass
    for _ in range(20):
        if port_wolny(8787):
            return
        time.sleep(0.25)
    raise RuntimeError(
        "Port 8787 jest dalej zajęty przez stary klaster. Zrestartuj kernel i odpal zadania od początku."
    )

zamknij_klaster()

zostaw = 8 * 1024**3
budzet = min(
    psutil.virtual_memory().total - zostaw,
    max(0, psutil.virtual_memory().available - 2 * 1024**3),
)
limit = 3 * 1024**3
rdzenie = max(1, min(os.cpu_count() - 2, budzet // limit))

client = Client(n_workers=int(rdzenie), threads_per_worker=1, memory_limit=limit)
print(f"workery: {rdzenie}, limit na worker: {limit / 1024**3:.0f} GiB")
print(f"zostaje co najmniej 8 GiB i {os.cpu_count() - rdzenie} rdzeni")
print(client.dashboard_link)
client

C:\Users\kaczo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\distributed\node.py:195: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 56580 instead
  warnings.warn(


workery: 6, limit na worker: 3 GiB
zostaje co najmniej 8 GiB i 10 rdzeni
http://127.0.0.1:56580/status


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:56580/status,
Dashboard: http://127.0.0.1:56580/status,Workers: 6
Total threads: 6,Total memory: 18.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:56583,Workers: 0
Dashboard: http://127.0.0.1:56580/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:56610,Total threads: 1
Dashboard: http://127.0.0.1:56619/status,Memory: 3.00 GiB
Nanny: tcp://127.0.0.1:56586,


In [4]:
import glob
import os
from datetime import datetime

import dask.dataframe as dd
from dask.distributed import wait

wszystkie = sorted(glob.glob(os.path.join("..", "lab_01", "data", "*.parquet")))
pliki = wszystkie[:rdzenie]

start = datetime.now()
ddf = client.persist(dd.read_parquet(pliki))
wait(ddf)
czas = datetime.now() - start

print(f"workery: {rdzenie}, limit na worker: 3 GiB")
print(f"wczytane pliki: {len(pliki)} z {len(wszystkie)}")
print(f"wiersze: {len(ddf)}")
print(f"czas: {czas}")
ddf.dtypes

C:\Users\kaczo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\distributed\node.py:195: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 56638 instead
  warnings.warn(


workery: 6, limit na worker: 3 GiB
wczytane pliki: 6 z 15
wiersze: 6884148
czas: 0:00:01.638349


sid                     int64
sid_profile             int64
post_id                string
profile_id              int64
date                   string
post_type               int64
description            string
likes                   int64
comments                int64
username               string
bio                    string
following               int64
followers               int64
num_posts               int64
is_business_account      bool
lang                   string
category               string
dtype: object

**Zadanie 4.** Operacje na ramce z zadania 3 (`ddf` musi już być w pamięci). Każda kończy się `.compute()`, więc czas obejmuje prawdziwe liczenie na klastrze. W tym czasie otwórz dashboard z `client.dashboard_link` i patrz na Task Stream.

In [5]:
from datetime import datetime

print(client.dashboard_link)

start = datetime.now()
top10 = ddf.groupby("username")["likes"].sum().nlargest(10).compute()
print("czas:", datetime.now() - start)
top10

http://127.0.0.1:56638/status
czas: 0:00:00.479385


username
instagram          29864166
lizakoshy          19217644
433                16457870
amandacerny        15019135
akshaykumar        13352324
maisie_williams    12808999
chiaraferragni     11709658
claireholt         10109849
danbilzerian        9253425
urvashirautela      9104884
Name: likes, dtype: int64

In [7]:
start = datetime.now()
polrocze = ddf[(ddf["date"] >= "2019-01-01") & (ddf["date"] < "2019-07-01")]
wynik = polrocze.compute()
print("czas:", datetime.now() - start)
print("wiersze:", len(wynik))
wynik.head()

2026-10-14 15:46:25,675 - distributed.nanny - WARNING - Restarting worker (status=Status.running)
2026-10-14 15:46:31,852 - distributed.nanny - WARNING - Restarting worker (status=Status.running)
2026-10-14 15:46:34,075 - distributed.nanny - WARNING - Restarting worker (status=Status.running)


KilledWorker: Attempted to run task ('repartitiontofewer-1f17b7c4fd7ccfb665ab42aad98fa4b5', 0) on 4 different workers, but all those workers died while running it. The last worker that attempt to run the task was tcp://127.0.0.1:55341. Inspecting worker logs is often a good next step to diagnose what went wrong. For more information see https://distributed.dask.org/en/stable/killed.html.

2026-10-14 15:46:40,035 - distributed.nanny - WARNING - Restarting worker (status=Status.running)


**Zadanie 5.**

**Przed optymalizacją.** Te same pliki co w zadaniu 3, typy zostają takie, jak w parquet.

In [3]:
import glob
import os
from datetime import datetime

import json
import socket
import time
from urllib.request import urlopen

import dask.dataframe as dd
import pandas as pd
import psutil
from dask.distributed import Client, get_client, wait

os.environ["PATH"] = r"C:\Program Files\Graphviz\bin" + os.pathsep + os.environ.get("PATH", "")

def port_wolny(port):
    with socket.socket() as s:
        s.settimeout(0.2)
        return s.connect_ex(("127.0.0.1", port)) != 0

def zamknij_klaster():
    try:
        client.shutdown()
    except Exception:
        pass
    try:
        get_client().shutdown()
    except Exception:
        pass
    if not port_wolny(8787):
        try:
            with urlopen("http://127.0.0.1:8787/json/identity.json", timeout=2) as resp:
                adres = json.load(resp)["address"]
            Client(adres).shutdown()
        except Exception:
            pass
    for _ in range(20):
        if port_wolny(8787):
            return
        time.sleep(0.25)
    raise RuntimeError(
        "Port 8787 jest dalej zajęty przez stary klaster. Zrestartuj kernel i odpal zadania od początku."
    )

zamknij_klaster()

zostaw = 8 * 1024**3
budzet = min(
    psutil.virtual_memory().total - zostaw,
    max(0, psutil.virtual_memory().available - 2 * 1024**3),
)
limit = 3 * 1024**3
rdzenie = max(1, min(os.cpu_count() - 2, budzet // limit))
client = Client(n_workers=int(rdzenie), threads_per_worker=1, memory_limit=limit)

pliki = sorted(glob.glob(os.path.join("..", "lab_01", "data", "*.parquet")))[:rdzenie]
print(f"pliki: {len(pliki)}, workery: {rdzenie}")
print(client.dashboard_link)

start = datetime.now()
ddf = client.persist(dd.read_parquet(pliki))
wait(ddf)
czas = datetime.now() - start

start = datetime.now()
top = ddf.groupby("username")["likes"].sum().nlargest(10).compute()
czas_top10 = datetime.now() - start

start = datetime.now()
ile = (
    ddf[(ddf["date"] >= "2019-01-01") & (ddf["date"] < "2019-07-01")]
    .map_partitions(len)
    .sum()
    .compute()
)
czas_h1 = datetime.now() - start
pamiec = ddf.memory_usage(deep=True).sum().compute()

print("typy przed optymalizacją")
print(ddf.dtypes.to_string())
print("\nładowanie:", czas)
print(f"pamięć: {pamiec / 1024**2:.0f} MiB")
print("top 10 likes:", czas_top10)
print("pierwsze półrocze 2019, wiersze:", ile, czas_h1)
print("\ntop 10")
print(top.to_string())
del ddf

pliki: 5, workery: 5
http://127.0.0.1:8787/status

typy przed optymalizacją
sid                     int64
sid_profile             int64
post_id                string
profile_id              int64
date                   string
post_type               int64
description            string
likes                   int64
comments                int64
username               string
bio                    string
following               int64
followers               int64
num_posts               int64
is_business_account      bool
lang                   string
category               string

ładowanie: 0:00:01.536015
pamięć: 2489 MiB
top 10 likes: 0:00:00.437873
pierwsze półrocze 2019, wiersze: 3963458 0:00:00.421202

top 10
username
instagram          29864166
lizakoshy          19217644
433                16457870
amandacerny        15019135
akshaykumar        13352324
maisie_williams    12808999
chiaraferragni     11709658
claireholt         10109849
danbilzerian        9253425
urvashirautela  

**Po optymalizacji.** Ten sam klaster i te same pliki. Typy z laboratorium 1 są nakładane na każdej partycji, potem wypisane poniżej.

In [2]:
typy = {
    "sid": "uint32",
    "sid_profile": "int32",
    "profile_id": "uint64",
    "post_type": "uint8",
    "likes": "uint32",
    "comments": "uint32",
    "following": "uint32",
    "followers": "uint32",
    "num_posts": "uint32",
    "lang": "category",
    "category": "category",
}

def optymalizuj(df):
    df = df.astype(typy)
    df["date"] = pd.to_datetime(df["date"])
    return df

meta = optymalizuj(dd.read_parquet(pliki).head(5)).iloc[:0]
start = datetime.now()
ddf_opt = dd.read_parquet(pliki).map_partitions(optymalizuj, meta=meta)
ddf_opt = client.persist(ddf_opt)
wait(ddf_opt)
czas = datetime.now() - start

start = datetime.now()
top_opt = ddf_opt.groupby("username")["likes"].sum().nlargest(10).compute()
czas_top10 = datetime.now() - start

start = datetime.now()
ile = (
    ddf_opt[(ddf_opt["date"] >= "2019-01-01") & (ddf_opt["date"] < "2019-07-01")]
    .map_partitions(len)
    .sum()
    .compute()
)
czas_h1 = datetime.now() - start
pamiec = ddf_opt.memory_usage(deep=True).sum().compute()

print("typy po optymalizacji")
print(ddf_opt.dtypes.to_string())
print("\nładowanie:", czas)
print(f"pamięć: {pamiec / 1024**2:.0f} MiB")
print("top 10 likes:", czas_top10)
print("pierwsze półrocze 2019, wiersze:", ile, czas_h1)
print("\ntop 10")
top_opt

typy po optymalizacji
sid                            uint32
sid_profile                     int32
post_id                        string
profile_id                     uint64
date                   datetime64[us]
post_type                       uint8
description                    string
likes                          uint32
comments                       uint32
username                       string
bio                            string
following                      uint32
followers                      uint32
num_posts                      uint32
is_business_account              bool
lang                         category
category                     category

ładowanie: 0:00:00.872928
pamięć: 2014 MiB
top 10 likes: 0:00:00.449903
pierwsze półrocze 2019, wiersze: 3963458 0:00:00.403331

top 10


username
instagram          29864166
lizakoshy          19217644
433                16457870
amandacerny        15019135
akshaykumar        13352324
maisie_williams    12808999
chiaraferragni     11709658
claireholt         10109849
danbilzerian        9253425
urvashirautela      9104884
Name: likes, dtype: uint32

**Zadanie 6.** Ta sama tablica co w przykładzie: `normal(5, 0.2)` o kształcie 20 000 na 20 000, liczona średnia po kolumnach (`mean(axis=0)`). Dla kilku podziałów na chunki pomiar idzie 10 razy, a w tabeli jest średnia. Scheduler to wątki tego procesu, nie klaster z Instagramem, żeby limit 3 GiB na worker nie ucinał dużego chunka.

In [ ]:
import time

import dask.array as da
import pandas as pd

rozmiar = (20_000, 20_000)
proby = 10
podzialy = [
    None,
    (1000, 1000),
    (2000, 2000),
    (4096, 4096),
    (5000, 5000),
    (10000, 10000),
]

wyniki = []
for chunks in podzialy:
    if chunks is None:
        tablica = da.random.normal(5, 0.2, size=rozmiar)
        opis = "automatyczny"
    else:
        tablica = da.random.normal(5, 0.2, size=rozmiar, chunks=chunks)
        opis = str(chunks)
    tablica.mean(axis=0).compute(scheduler="threads")
    czasy = []
    for _ in range(proby):
        start = time.perf_counter()
        tablica.mean(axis=0).compute(scheduler="threads")
        czasy.append(time.perf_counter() - start)
    srednia = sum(czasy) / len(czasy)
    wyniki.append(
        {
            "podział": opis,
            "chunk": str(tuple(tablica.chunksize)),
            "liczba chunków": tablica.npartitions,
            "średni czas s": round(srednia, 3),
        }
    )
    print(f"{opis}: chunk {tuple(tablica.chunksize)}, {tablica.npartitions} chunków, {srednia:.3f} s")

pd.DataFrame(wyniki)

automatyczny: chunk (4096, 4096), 25 chunków, 1.005 s
(1000, 1000): chunk (1000, 1000), 400 chunków, 0.985 s
(2000, 2000): chunk (2000, 2000), 100 chunków, 0.994 s
(4096, 4096): chunk (4096, 4096), 25 chunków, 1.012 s
(5000, 5000): chunk (5000, 5000), 16 chunków, 0.997 s


**Wniosek.** Automatyczny podział na tej maszynie wybiera chunk `4096×4096` (25 kawałków) i średnią liczy w około **1,0 s**. Ręczny podział z przykładu, `2000×2000`, jest praktycznie tak samo szybki. Najkrócej wyszło `1000×1000` (400 chunków, około **0,93 s**), czyli da się zejść poniżej automatu, ale tylko o kilka setnych sekundy. `10000×10000` (4 chunki) zajmuje około **2,0 s**: za mało kawałków, żeby obciążyć rdzenie. Domyślny podział nie jest najszybszy co do setnej, ale siedzi w tym samym dobrym zakresie co rozsądne podziały ręczne. Wyraźnie gorzej jest dopiero wtedy, gdy chunków jest za mało.